# R41 · Mine useful training examples, not more false boxes

**Choose a GPU runtime (L4 or T4 is fine), then Runtime → Run all.** CPU is supported as a slower alternative. No training is performed. This is a new inference run, not a repeat of R39.

R40 established 643 matches, 11 misses and 85 unmatched predictions on accepted R38 (PR123 with post-clip NMS). There are 320 existing, separately archived upstream frames, but their original `white_box` labels have not been qualified under our current visible-front convention. We will use the frozen PR123 checkpoint to mine differences, revisit the three earlier pilot images, and select nine additional full images for a bounded review. Neither source annotations nor model proposals are ground truth in this pool.

The notebook verifies the exact upstream image/label bytes and PR123 checkpoint before use. It never trains, edits existing annotations, changes thresholds, promotes a model, or declares the pool an independent test set. It returns `baomai-front-mining-r41-results.zip`. Please send that ZIP back for the next annotation decision and controlled GPU training notebook.

[PR126 / R40](https://github.com/Siuuuers/dl/pull/126) · [Current boundary guidance](https://github.com/Siuuuers/dl/blob/5852d50e49043cd38406c76c54c7ce688beeb462/docs/annotation-boundaries-r33.md) · [Earlier pilot](https://github.com/Siuuuers/dl/blob/5852d50e49043cd38406c76c54c7ce688beeb462/docs/additional-annotation-pilot-2026-11-07.md)

In [ ]:
# 1. Configuration and immutable input verification (standard library)
import csv, hashlib, io, json, math, os, re, shutil, sys, time, urllib.request, zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter

R40 = '5852d50e49043cd38406c76c54c7ce688beeb462'
UPSTREAM = '9a6bf2432582af517d789c94eab2ac27e03a304b'
PR123 = '6707967049c548396745e66127691028c17c20a4'
WEIGHT_SHA = '51d8a60f95b1b76097513d8b07732de5d5deed5e6e17251a28ddd28e6be1ebc1'
POOL_SHA = 'b28bcce5fffe91f3e486ff11ec3085a67c868fe09e65c14788bc6fe494af4dd0'
MODEL_ARCHIVE_SHA = '41039efb8643f97fc35beb29c71679fd10d0430cdaad43fafa790849d10dac58'
REPO = f'https://raw.githubusercontent.com/Siuuuers/dl/{R40}/'
RETURN = f'https://raw.githubusercontent.com/Siuuuers/dl/{PR123}/results/gpu/baomai-front-review-r37-l4-001/'
POOL_URL = f'https://media.githubusercontent.com/media/an94-doll/embedding_vision/{UPSTREAM}/box_in_basket/white_box_pose_reviewed_320_images_20261028_145012.zip'
SCRIPTS = {
    'evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0',
    'pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4',
    'pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018',
    'predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6',
}
PARTS = [
    ('00', '3989dbd3e4c9e6e09a3f474e250f51b968a6195255c0aa96cafb4d8b3ac523a6', 'd0c36aa7b1ca367905b0a9fccc282f8749ab8206510437b64809034844612b2f'),
    ('01', '579429ef94843642bd46c7a98a88e8f7703db14e958e104af64e244cee42e3ba', 'c7f30fc73fa8bdb194f53225cc1609275b91ffaca1f4ee43c2a4c17aa2706ed5'),
]
PILOT_IDS = ('frame_0001', 'frame_0160', 'frame_0383')
PILOT = 'data/candidates/additional-pilot-2026-11-07/'
WORK = Path(os.environ.get('BAOMAI_R41_WORK', '/content/baomai-r41' if Path('/content').is_dir() else 'baomai-r41')).resolve()
CACHE = WORK / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)

def digest(data):
    return hashlib.sha256(data).hexdigest()

def verify_file(path, expected, size=None):
    if (size is not None and path.stat().st_size != size) or digest(path.read_bytes()) != expected:
        raise ValueError(f'Hash/size mismatch (existing bytes preserved): {path}')
    return path

def store(path, data, expected):
    if digest(data) != expected:
        raise ValueError(f'Unexpected bytes for {path}')
    if path.exists():
        return verify_file(path, expected)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_bytes(data)
    tmp.replace(path)
    return path

def download(url, path, expected, size=None):
    if path.exists():
        return verify_file(path, expected, size)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.download')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=120) as src, tmp.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            verify_file(tmp, expected, size)
            tmp.replace(path)
            return path
        except Exception:
            tmp.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

def archive_names(bundle):
    names = [item.filename for item in bundle.infolist()]
    if len(names) != len(set(names)):
        raise ValueError('Duplicate ZIP paths')
    for item in bundle.infolist():
        p = PurePosixPath(item.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in item.filename or (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f'Unsafe ZIP member: {item.filename}')
    return set(names)

def read_member(bundle, suffix, names):
    matches = [n for n in names if n == suffix or n.endswith('/' + suffix)]
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one ZIP member for {suffix}, got {matches}')
    return bundle.read(matches[0])

def acquire_inputs():
    source_dir = CACHE / 'source'
    for name, sha in SCRIPTS.items():
        download(REPO + 'scripts/' + name, source_dir / name, sha)
    intake_file = download(REPO + 'docs/additional-frame-intake-2026-11-07.json', CACHE / 'intake.json',
                           '6754ce0994dfb4f2587117bd213df9896283f87b4dacf0d7a714562d41e8a8c6')
    intake = json.loads(intake_file.read_text())
    assert intake['source']['commit'] == UPSTREAM and intake['source']['zip_sha256'] == POOL_SHA
    records = sorted(intake['reviewed320_records'], key=lambda r: r['image'])
    if len(records) != 320 or len({r['image'] for r in records}) != 320 or len({r['image_sha256'] for r in records}) != 320:
        raise ValueError('Expected 320 distinct source frames')
    pool = download(POOL_URL, CACHE / 'reviewed320.zip', POOL_SHA, 43424432)
    with zipfile.ZipFile(pool) as z:
        names = archive_names(z)
        for r in records:
            for key in ('image', 'label'):
                rel = r[key]
                if not re.fullmatch(r'(images/frame_\d+\.jpg|labels/frame_\d+\.txt)', rel):
                    raise ValueError(f'Unexpected source member {rel}')
                store(CACHE / 'pool' / rel, read_member(z, rel, names), r[key + '_sha256'])
    # Reassemble the exact original PR123 result archive, not an arbitrary uploaded .pt.
    full = CACHE / 'pr123.zip'
    if full.exists():
        verify_file(full, MODEL_ARCHIVE_SHA, 39799383)
    else:
        tmp = full.with_suffix('.assembling')
        with tmp.open('wb') as out:
            for number, wrapper_sha, chunk_sha in PARTS:
                name = f'baomai-front-review-r37-results.zip.part{number}'
                wrapper = download(RETURN + name + '.zip', CACHE / (name + '.zip'), wrapper_sha)
                with zipfile.ZipFile(wrapper) as z:
                    chunk = read_member(z, name, archive_names(z))
                    if digest(chunk) != chunk_sha:
                        raise ValueError('PR123 part hash differs')
                    out.write(chunk)
        verify_file(tmp, MODEL_ARCHIVE_SHA, 39799383)
        tmp.replace(full)
    with zipfile.ZipFile(full) as z:
        names = archive_names(z)
        weights = store(CACHE / 'pr123-candidate.pt', read_member(z, 'candidate/fit/weights/last.pt', names), WEIGHT_SHA)
        manifest = json.loads(read_member(z, 'candidate/checkpoint-manifest.json', names))
        assert manifest['checkpoint']['sha256'] == WEIGHT_SHA and manifest['kpt_shape'] == [2, 3]
    # The three old proposed label files are separate from original upstream labels.
    # Verify the manifest against its immutable Git blob identity before reading its label hashes.
    pilot_path = CACHE / 'pilot-manifest.json'
    if pilot_path.exists():
        pilot_bytes = pilot_path.read_bytes()
    else:
        with urllib.request.urlopen(REPO + PILOT + 'manifest.json', timeout=120) as f:
            pilot_bytes = f.read()
    blob = hashlib.sha1(f'blob {len(pilot_bytes)}\0'.encode() + pilot_bytes).hexdigest()
    if blob != '830b4e45d9a2aceedd7bcd7a22731b57b41070be':
        raise ValueError('Pilot manifest Git blob identity differs')
    pilot = json.loads(pilot_bytes)
    assert pilot['source_commit'] == UPSTREAM and set(pilot['selected_image_ids']) == set(PILOT_IDS) and not pilot['eligible_image_ids']
    store(CACHE / 'pilot-manifest.json', pilot_bytes, digest(pilot_bytes))
    for r in pilot['images']:
        assert r['disposition'] == 'quarantined'
        download(REPO + PILOT + r['candidate_label'], CACHE / 'pilot' / (r['id'] + '.txt'), r['candidate_label_sha256'])
    return records, weights, pilot

records, weights, pilot = acquire_inputs()
print(f'Verified {len(records)} original image/label pairs, PR123 candidate weights, R40 scripts and three quarantined pilot proposals.')

## Fresh predictions

The existing R40 functions supply the inference settings and post-clipping filter. Confidence = 0.13; ordinary and additional NMS IoU = 0.70; keypoint display = 0.50. The source-label comparison is only a review-priority signal. Hardware can affect floating-point results; record the actual runtime rather than assuming exact agreement with earlier CPU inference.

In [ ]:
# 2. Install only the required model package; use the existing Colab torch runtime.
import importlib.metadata as metadata
import subprocess
try:
    installed = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    installed = None
if installed != '8.4.182':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.182'], check=True)
import torch
from PIL import Image, ImageDraw, ImageFont
sys.path.insert(0, str(CACHE / 'source'))
import evaluate_pose as evaluator
import predict_pose_folder as runner
from pose_scoring import score_image
for module in (evaluator, runner, sys.modules['pose_checkpoint'], sys.modules['pose_scoring']):
    if Path(module.__file__).resolve().parent != (CACHE / 'source').resolve():
        raise RuntimeError('Restart the runtime: an unpinned repository module is already loaded.')
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
CONF, ENDPOINT_CONF = 0.13, 0.50
if DEVICE == 'cpu':
    print('CPU runtime detected; this is supported but GPU is recommended for 320 frames.')

# Each execution has a new result directory, so previous output is not overwritten.
RUN = WORK / ('run-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN.mkdir(parents=True, exist_ok=False)
first_rgb, _ = runner.decode_image(CACHE / 'pool' / records[0]['image'])
predict, runtime = runner.prepare_predictor(weights, DEVICE, 4, first_rgb, conf=CONF, nms_iou=0.70)
if DEVICE != 'cpu':
    runtime['device_name'] = torch.cuda.get_device_name(0)

def process_frame(r, predictor):
    image = CACHE / 'pool' / r['image']
    rgb, meta = runner.decode_image(image)
    if (rgb.width, rgb.height) != (r['width'], r['height']) or meta['image_sha256'] != r['image_sha256']:
        raise ValueError(f'Image mismatch: {r["image"]}')
    label_path = verify_file(CACHE / 'pool' / r['label'], r['label_sha256'])
    provisional = evaluator.parse_pose_labels(label_path.read_text(), rgb.width, rgb.height, r['label'])
    if len(provisional) != r['object_count']:
        raise ValueError(f'Source row count differs: {r["label"]}')
    result = predictor(rgb)
    described = runner.describe_predictions(result['predictions'], rgb.width, rgb.height, ENDPOINT_CONF)
    kept, audit = runner.postclip_nms(described)
    # Score against old, unqualified source rows for triage only. Do not use as accuracy evidence.
    comparison = score_image(provisional, kept, rgb.width, rgb.height, 0.50, ENDPOINT_CONF)
    missing_source_rows = [i + 1 for i in comparison['unmatched_gold_indices']]
    extra_ids = [kept[i]['primary_prediction_id'] for i in comparison['unmatched_prediction_indices']]
    duplicate_ids = [kept[i]['primary_prediction_id'] for i in comparison['duplicate_prediction_indices']]
    partial = sum(any(p[2] == 0 for p in obj['keypoints']) for obj in provisional)
    matched = [{'source_row': m['gold_index'] + 1, 'primary_prediction_id': kept[m['prediction_index']]['primary_prediction_id'],
                'iou': m['iou']} for m in comparison['matches']]
    # Equal weight per discrepancy; partial count is a tie-breaker, not an accuracy measure.
    priority = len(extra_ids) + len(missing_source_rows)
    summary = {'image_id': Path(r['image']).stem, 'source_image_sha256': r['image_sha256'],
               'source_label_sha256': r['label_sha256'], 'source_rows': len(provisional), 'source_partial_rows': partial,
               'primary_outputs': len(described), 'filtered_outputs': len(kept), 'suppressed': len(audit['suppressed_predictions']),
               'source_unmatched_rows': len(missing_source_rows), 'source_unmatched_outputs': len(extra_ids),
               'source_duplicate_subset': len(duplicate_ids), 'priority': priority}
    detail = {**summary, 'source_unmatched_row_ids': missing_source_rows, 'source_unmatched_primary_ids': extra_ids,
              'source_duplicate_primary_ids': duplicate_ids, 'source_matches': matched,
              'predictions': kept, 'postclip_nms': audit, 'stage_ms': result['stage_ms'],
              'predict_call_wall_ms': result['predict_call_wall_ms']}
    return summary, detail

summaries = []
with (RUN / 'predictions.jsonl').open('x', encoding='utf-8') as stream:
    for index, record in enumerate(records, 1):
        summary, detail = process_frame(record, predict)
        stream.write(json.dumps(detail, allow_nan=False) + '\n')
        stream.flush()
        summaries.append(summary)
        if index % 20 == 0 or index == len(records):
            print(f'{index}/{len(records)} frames inferred', flush=True)
assert len(summaries) == 320
print('Completed all 320 new inference inputs. No training performed.')

## Select a small full-image review batch

Keep the three previously quarantined pilots. From the remaining 317 images, choose nine new review frames by disagreement count, then partial-front count, then stable frame ID. Divide the ordered remaining filenames into nine equal-sized buckets, selecting one per bucket. Prefer candidates at least five filename-index numbers apart from the selected frames; if a bucket has no such candidate, select its highest-ranked image and record the fallback. This is a crude diversity guard, **not** proof of capture-session independence. No image is selected for training automatically.

In [ ]:
# 3. Review selection, original and predicted overlays, and separate provisional labels.
def frame_number(image_id):
    return int(image_id.split('_')[-1])

def choose_review(rows, pilot_ids=PILOT_IDS, buckets=9):
    by_id = {r['image_id']: r for r in rows}
    if len(by_id) != len(rows) or not set(pilot_ids) <= set(by_id):
        raise ValueError('Missing pilot or duplicate image ID')
    chosen = [{**by_id[name], 'selection_reason': 'revisit_quarantined_pilot', 'bucket': None,
               'spacing_fallback': False} for name in pilot_ids]
    remaining = sorted((r for r in rows if r['image_id'] not in pilot_ids), key=lambda r: frame_number(r['image_id']))
    if len(remaining) < buckets:
        raise ValueError('Not enough frames for all review buckets')
    for b in range(buckets):
        group = remaining[b*len(remaining)//buckets:(b+1)*len(remaining)//buckets]
        ranked = sorted(group, key=lambda r: (-r['priority'], -r['source_partial_rows'], r['image_id']))
        spaced = [r for r in ranked if all(abs(frame_number(r['image_id']) - frame_number(c['image_id'])) >= 5 for c in chosen)]
        selected = (spaced or ranked)[0]
        chosen.append({**selected, 'selection_reason': 'source_disagreement_mining', 'bucket': b + 1,
                       'spacing_fallback': not bool(spaced)})
    return chosen

def write_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)

def source_overlay(rgb, objects, path):
    overlay = rgb.copy()
    draw = ImageDraw.Draw(overlay)
    font = ImageFont.load_default(size=13)
    for i, obj in enumerate(objects, 1):
        box = obj['bbox_xyxy']
        draw.rectangle(box, outline='#00c8e5', width=1)
        draw.text((max(0, box[0]), max(0, box[1]-14)), str(i), fill='white', stroke_width=1, stroke_fill='black', font=font)
        for point in obj['keypoints']:
            x,y,v = point
            if v == 2 and 0 <= x < rgb.width and 0 <= y < rgb.height:
                draw.ellipse((x-3,y-3,x+3,y+3),fill='#00c8e5',outline='black')
    overlay.save(path)

selected = choose_review(summaries)
assert len(selected) == 12 and len({r['image_id'] for r in selected}) == 12
write_csv(RUN / 'all-frame-priorities.csv', sorted(summaries, key=lambda r: (-r['priority'], -r['source_partial_rows'], r['image_id'])))
write_csv(RUN / 'review-queue.csv', selected)
index = {Path(r['image']).stem: r for r in records}
# Load only the 12 selected prediction records for illustration.
selected_ids = {r['image_id'] for r in selected}
details = {}
with (RUN / 'predictions.jsonl').open() as f:
    for line in f:
        d = json.loads(line)
        if d['image_id'] in selected_ids:
            details[d['image_id']] = d
assert set(details) == selected_ids
review_dir = RUN / 'review'
review_dir.mkdir()
for item in selected:
    name = item['image_id']; r = index[name]; d = details[name]
    dest = review_dir / name; dest.mkdir()
    rgb, _ = runner.decode_image(CACHE / 'pool' / r['image'])
    shutil.copyfile(CACHE / 'pool' / r['image'], dest / 'original.jpg')
    shutil.copyfile(CACHE / 'pool' / r['label'], dest / 'source.txt')
    rgb.save(dest / (name + '.png'))
    # The paired TXT is provisional, for the existing browser editor; no automatic acceptance.
    proposed = CACHE / 'pilot' / (name + '.txt') if name in PILOT_IDS else CACHE / 'pool' / r['label']
    shutil.copyfile(proposed, dest / (name + '.txt'))
    obj = evaluator.parse_pose_labels((dest / 'source.txt').read_text(), rgb.width, rgb.height)
    source_overlay(rgb, obj, dest / 'source-overlay.png')
    if name in PILOT_IDS:
        previous = evaluator.parse_pose_labels(proposed.read_text(), rgb.width, rgb.height)
        source_overlay(rgb, previous, dest / 'pilot-proposal-overlay.png')
        p = next(p for p in pilot['images'] if p['id'] == name)
        (dest / 'pilot-review.json').write_text(json.dumps(p, indent=2) + '\n')
    runner._save_overlay(rgb, d['predictions'], dest / 'prediction-overlay.png')
    (dest / 'comparison.json').write_text(json.dumps(d, indent=2, allow_nan=False) + '\n')

# Local, self-contained gallery. The original JPEG and PNG are both retained;
# the PNG is a lossless export of decoded source pixels, not a new camera capture.
import html
parts = ['<!doctype html><meta charset="utf-8"><title>R41 review</title>',
         '<style>body{font:16px system-ui;max-width:1500px;margin:auto;padding:16px;color:#182535} section{border-top:1px solid #ccc;margin-top:24px;padding-top:12px} .panels{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:12px} img{width:100%;height:auto} a{color:#1263a0}</style>',
         '<h1>R41 full-image review</h1><p>Old source labels (cyan) are unqualified. Predictions are separate proposals. No labels are approved. Open original frames, check all visible fronts, then amend the paired PNG/TXT if supported.</p>']
for i, item in enumerate(selected, 1):
    name = item['image_id']; base = 'review/' + name + '/'
    parts.append(f'<section><h2>{i}. {html.escape(name)}</h2><p>{item["selection_reason"]}: {item["source_unmatched_outputs"]} unmatched predictions and {item["source_unmatched_rows"]} unmatched old rows (not established errors).</p><div class="panels">')
    panels = [('Original', name + '.png'), ('Old source rows', 'source-overlay.png')]
    if name in PILOT_IDS: panels.append(('Quarantined pilot proposal', 'pilot-proposal-overlay.png'))
    panels.append(('PR123 + R40 outputs', 'prediction-overlay.png'))
    for label, file in panels:
        parts.append(f'<div><b>{label}</b><a href="{base}{file}"><img src="{base}{file}" alt="{label}"></a></div>')
    parts.append(f'</div><p><a href="{base}{name}.txt">Provisional editable TXT</a> · <a href="{base}comparison.json">Exact proposal and old-row comparison</a></p></section>')
(RUN / 'review.html').write_text('\n'.join(parts), encoding='utf-8')
print('Selected for complete-image review:', ', '.join(r['image_id'] for r in selected))

## Return one ZIP

The returned package includes all 320 new inference records (with original prediction IDs and suppression audits), the full priority table, the 12-image review queue, full originals and overlays for the selected images, prior pilot evidence, and the actual runtime/immutable input information. The provisional editor TXT files are deliberately separate from source labels and model proposals. Return this ZIP to ChatGPT/GitHub for full-image annotation review before any new training.

In [ ]:
# 4. Package the complete new result.
receipt = {
    'kind': 'r41-unqualified-upstream-review-mining', 'source_revision': R40,
    'upstream_revision': UPSTREAM, 'pool_archive_sha256': POOL_SHA,
    'model_return_revision': PR123, 'model_return_archive_sha256': MODEL_ARCHIVE_SHA,
    'checkpoint_sha256': WEIGHT_SHA, 'source_scripts_sha256': SCRIPTS,
    'intake_sha256': digest((CACHE/'intake.json').read_bytes()),
    'pilot_manifest_sha256': digest((CACHE/'pilot-manifest.json').read_bytes()),
    'runtime': runtime, 'inference': {**evaluator.INFERENCE, 'conf': CONF, 'endpoint_confidence': ENDPOINT_CONF,
                                       'postclip_nms_iou': 0.70, 'source_comparison_iou': 0.50},
    'fresh_inference': True, 'completed_images': len(summaries), 'training': False,
    'new_approved_training_images': 0, 'automatic_promotion': False,
    'selected_review_ids': [r['image_id'] for r in selected],
    'selection': '3 quarantined pilots + 9 filename-order bucket selections; rank by old-label disagreements and partial rows',
    'limits': ['Old upstream labels are not qualified under current visible-front policy; no accuracy claim.',
               'Frame-number spacing is not capture-session independence.',
               'Inference uses the hash-verified immutable returned checkpoint and existing R40 predictor; the full historical training dataset is not revalidated.',
               'No model-generated point or box is automatically approved for training.'],
}
(RUN / 'receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False) + '\n')
(RUN / 'README.txt').write_text('R41 review mining, not model training. Open review.html after extracting the ZIP. Review the full selected frames under current cassette-front rules. Original labels and predicted outputs are separate; paired PNG/TXT is a provisional editing copy only. No new supervision is approved. Return the ZIP before running a training comparison.\n', encoding='utf-8')
files = sorted(p for p in RUN.rglob('*') if p.is_file())
manifest = {p.relative_to(RUN).as_posix(): {'sha256': digest(p.read_bytes()), 'bytes': p.stat().st_size} for p in files}
(RUN / 'inventory.json').write_text(json.dumps(manifest, indent=2) + '\n')
export = RUN.parent / ('export-' + RUN.name)
export.mkdir(exist_ok=False)
archive = export / 'baomai-front-mining-r41-results.zip'
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(q for q in RUN.rglob('*') if q.is_file()):
        z.write(p, p.relative_to(RUN).as_posix())
with zipfile.ZipFile(archive) as z:
    names = archive_names(z)
    assert names == set(manifest) | {'inventory.json'} and z.testzip() is None
    for name, entry in manifest.items():
        b = z.read(name)
        assert digest(b) == entry['sha256'] and len(b) == entry['bytes']
print(f'Complete: {len(summaries)} predictions sets; {len(selected)} full review frames; {archive.stat().st_size:,} bytes.')
print('SHA-256:', digest(archive.read_bytes()))
print('No new training or promotion performed.')
try:
    from google.colab import files as colab_files
    colab_files.download(str(archive))
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))